empecemos por algo simple: mapa de cuanta gente por AGSN en SP en choropleth y quizas otras infos. Hacerlo en leafmap pa cambiar de layers. 

#elegir imagen de nightlights y ver un clip de cuales de esos caen en los poligonos de inundacion de NOAA- Pueblos dentro de zonas de riesgo de inundacion RD

In [ ]:
---
layout: post
title: Welcome to Jekyll!
featured: true
tags: [frontpage, jekyll, blog]
image: '/images/welcome.jpg'
---

In [1]:
#!pip install leafmap
# Important library for many geopython libraries
#!pip install gdal-bin python-gdal python3-gdal 
# Install rtree - Geopandas requirment
#!pip install python3-rtree 
# Install Geopandas
#pip install geopandas
# Install descartes - Geopandas requirment
#!pip install descartes
#!pip install mapclassify

In [2]:
import pandas as pd
import numpy as np
import leafmap # for interactive mapping
import geopandas as gpd
#import matplotlib.pyplot as plt
import folium

Importar sao paulo data

In [3]:
path_shp = r'C:\Users\Isabela\Downloads\AGSN_2019\AGSN_2019.shp'
shp_mask = gpd.read_file(
    path_shp
)

shp = gpd.read_file(
    path_shp,
    mask=shp_mask[shp_mask.UF == 'São Paulo'],
)

# create id column
shp['ID'] = np.arange(start=1, stop=len(shp.UF)+1, step=1)

shp = shp[['ID', 'SUM_EDOC', 'geometry']]#[[]] returns dataframe

#create data only
data = pd.DataFrame(shp)

shp.head()

#path_data = r'C:\Users\Isabela\Downloads\AGSN_2019.csv'
#data = pd.read_csv(path_data) # csv and shp are not in same order and dont have objectid

,ID,SUM_EDOC,geometry
0,1,1005,"POLYGON ((-46.61809 -23.65966, -46.61772 -23.6..."
1,2,432,"POLYGON ((-46.49638 -23.70153, -46.49638 -23.7..."
2,3,985,"POLYGON ((-46.66907 -23.68819, -46.66858 -23.6..."
3,4,252,"POLYGON ((-46.57125 -23.51516, -46.57126 -23.5..."
4,5,90,"POLYGON ((-46.45861 -23.63947, -46.45933 -23.6..."


Crear Choropleth

In [4]:
# folium, https://vverde.github.io/blob/interactivechoropleth.html
x_map=shp.centroid.x.mean()
y_map=shp.centroid.y.mean()

x_map, y_map

# define basemap
map = folium.Map(location=[y_map, x_map], zoom_start=13, tiles=None)
folium.TileLayer('CartoDB positron', name='Light Map', control=False).add_to(map)
map

# define threshold of population parameter legend
folium.Choropleth(
    geo_data=shp, 
    name='Choropleth', 
    data=data,
    columns=['ID', 'SUM_EDOC'],
    key_on='feature.properties.ID',
    fill_color='YlGnBu',
    fill_opacity=1,
    line_opacity=0.2,
    legend_name='Habitants per Favela',
).add_to(map)


# then load transparent layer to make choropleth interactive
folium.features.GeoJson(shp,  name='Labels',
               style_function=lambda x: {'color':'transparent','fillColor':'transparent','weight':0},
                tooltip=folium.features.GeoJsonTooltip(fields=['SUM_EDOC'],
                                              aliases = ['Population'],
                                              labels=True,
                                              sticky=False
                                             )
                       ).add_to(map)

map

*That's the game for today!*